# 🚀 업비트 자동 매매 시스템 v4.0

## BTC·SOL·ETH·XRP | 1시간봉 | LightGBM·XGBoost·GRU·앙상블 | Optuna | 리스크관리 v4

### v4 핵심 개선사항 (v3 대비)
- ✅ **타임프레임 상향**: 5분봉 → **1시간봉** (수수료·슬리피지 대비 신호 품질 향상)
- ✅ **예측 horizon 상향**: 6봉(30분) → **12봉(12시간)** (충분한 수익 여지 확보)
- ✅ **레이블 기준 강화**: ±0.5% → **±1.5%** (횡보 구간 필터, 노이즈 감소)
- ✅ **학습률 스케줄러**: CosineAnnealing + WarmUp (DL 모델 수렴 품질 개선)
- ✅ **클래스 불균형 보정**: `class_weight` + focal loss (횡보 편향 제거)
- ✅ **피처 정규화 수정**: 테스트 셋 leakage 제거 (train fit → test transform only)
- ✅ **Walk-Forward 실구현**: expanding window, 6-fold
- ✅ **백테스트 버그 수정**: exec_price 경계, hold_bar 이중관리, 피라미딩 선언 불일치
- ✅ **AutoTrader 클래스명 통일**: AutoTraderV3
- ✅ **손절 최소 보유 분리**: 손절은 3봉 이후, 신호 매도는 MIN_HOLD_BARS 유지


In [ ]:
# 원본 마크다운은 보존하며 아래 실행 코드는 v5 GPU·현물 모의 전용입니다.
import sys
from pathlib import Path
import json
import engine
print('원본 v4 설명은 이력입니다. 실제 실행은 v5 XGBoost CUDA와 공개 호가 모의투자입니다.')


In [ ]:
# CPU는 수집과 전처리, GPU는 모델 적합에 사용합니다.
PROJECT_ROOT = Path(engine.__file__).resolve().parent
CONFIG_PATH = PROJECT_ROOT / 'engine-config.json'
print('프로젝트:', PROJECT_ROOT)


In [ ]:
# 예정 시작 전에는 모의체결도 하지 않습니다.
config = engine.load_config(CONFIG_PATH)
print(json.dumps(config, ensure_ascii=False, indent=2))


In [ ]:
# 최근 수집만 한 차례 수행합니다. 전체 이력은 별도 collector --backfill 프로세스가 이어갑니다.
collection_summary = engine.collect_cycle(config)
print(json.dumps(collection_summary, ensure_ascii=False, indent=2))


In [ ]:
# 완료 시각을 기준으로 결합한 피처를 확인합니다.
store = engine.CandleStore(Path(config['data_dir']) / 'candles.sqlite3')
audit = engine.ExperimentAudit(Path(config['data_dir']) / 'experiment_audit.sqlite3')
consumed = audit.consumed_periods()
plan = engine.experiment_plan(config, live=True, consumed=consumed)
market = config['markets'][0]
preview = engine.load_market_features(store, market, config, plan['train'][0], plan['validation'][1])
print(preview.tail())


In [ ]:
# 미래 라벨 끝 시각과 경계 간격을 확인합니다.
training = config['training']
labelled = engine.label_frame(preview, training['horizon'], training['label_threshold'], config['paper']['fee'], config['paper']['slippage'])
complete = labelled.dropna(subset=engine.feature_columns(labelled) + ['label', 'label_end'])
splits = engine.research_splits(complete, plan, config, consumed)
print({key: len(value) for key, value in splits.items()})


In [ ]:
# 실제 CUDA 적합 설정이 확인되지 않으면 중단합니다.
gpu_evidence = engine.check_gpu()
print(json.dumps(gpu_evidence, ensure_ascii=False, indent=2))


In [ ]:
# 다음 연구 후보를 사용하며 적합·튜닝·확률 보정·정책 선정·후속 평가를 분리합니다.
with engine.process_lock(PROJECT_ROOT / 'work/service/trainer.lock'):
    report = engine.train_cycle(config, final=False, live=True)
print('실험:', report['run'])


In [ ]:
# 시험 구간은 후보 선택이나 조기 종료에 사용하지 않습니다.
print(json.dumps(report['periods'], ensure_ascii=False, indent=2))
print(json.dumps(report['locked_test'], ensure_ascii=False, indent=2))


In [ ]:
# 수익·완료 거래 근거와 승격 비교를 통과한 경우에만 포인터를 교체합니다.
manifest = None
if (Path(config['models_dir']) / 'current.json').is_file():
    directory, manifest = engine.load_current(config)
    print('현재 모델:', directory, '운용 보류 사유:', engine.deployment_guard(manifest))
else:
    print('운용 보류: 승격을 통과한 모델이 없습니다.')


In [ ]:
# 후보 정책 진단은 현재 운용 모델과 분리하여 확인합니다.
print(json.dumps(report['research'], ensure_ascii=False, indent=2))
print(json.dumps({m: p['rule'] for m, p in report['markets'].items()}, ensure_ascii=False, indent=2))


In [ ]:
# 현물만 보유하며 차입과 공매도를 허용하지 않습니다.
paper_settings = config['paper']
print(json.dumps(paper_settings, ensure_ascii=False, indent=2))


In [ ]:
# 정책 선정 이후 구간의 후속 평가이며 반복 연구를 독립 시험 성과로 표현하지 않습니다.
print(json.dumps(report['validation'], ensure_ascii=False, indent=2))


In [ ]:
# 시작 시각 이전에는 대기 상태만 저장합니다. 외부 알림과 실제 주문은 없습니다.
paper_result = engine.paper_cycle(config)
print({k: paper_result.get(k) for k in ('status', 'equity', 'net_return', 'MDD')})


In [ ]:
# 95%는 목표이며 미달/표본 수/다수 클래스 기준선도 함께 표시합니다.
for market, result in report['markets'].items():
    print(market, json.dumps(result['validation'], ensure_ascii=False))
print('19~20일 UTC 잠금 시험은 이미 소비되었으므로 재평가하지 않습니다.')
